# Week 4 Day 5 — Save PCA Outputs

Persists two files that Week 6 (hedging) will load directly:

| file | contents | used for |
|------|----------|----------|
| `pca_loadings.parquet` | sign-corrected loading vectors + variance ratios | factor-exposure calculation: `KRD @ PC_loading` |
| `factor_scores.parquet` | daily bp scores for every PC | regime labelling in Week 7 |

Nothing new is computed here — this is purely a persistence step.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from termstructure.pca.decomposition import save_pca_outputs
from termstructure.pca.panel import DEFAULT_MATURITIES

## 1. Save outputs

In [ ]:
info = save_pca_outputs()

print(f"Components saved : {info['n_components']}")
print(f"Score rows saved : {info['n_score_rows']}")
print(f"PC1+PC2+PC3 var  : {info['var_3pc']:.1%}")
print(f"Loadings path    : {info['loadings_path']}")
print(f"Scores path      : {info['scores_path']}")

## 2. Loadings parquet

One row per PC. The `y_*` columns are the sign-corrected loading weights —
Week 6 will dot-product these with a bond's key-rate durations to get its
factor exposure.

In [ ]:
loadings = pd.read_parquet('../data/processed/pca_loadings.parquet')
loadings.set_index('pc', inplace=True)
print(f'Shape: {loadings.shape}  (n_components × (var_ratio + n_maturities))')
loadings.round(4)

## 3. Loading heatmap — quick sanity check

PC1 should be flat, PC2 monotonic, PC3 U-shaped.
Rows 4–8 should look like noise.

In [ ]:
mat_cols = [c for c in loadings.columns if c.startswith('y_')]
L = loadings[mat_cols].to_numpy()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(L, aspect='auto', cmap='RdBu_r', vmin=-0.8, vmax=0.8)
plt.colorbar(im, ax=ax, label='Loading')
ax.set_xticks(range(len(DEFAULT_MATURITIES)))
ax.set_xticklabels([f'{m}Y' for m in DEFAULT_MATURITIES])
ax.set_yticks(range(len(L)))
ax.set_yticklabels([f'PC{i+1}  ({loadings["var_ratio"].iloc[i]:.1%})' for i in range(len(L))])
ax.set_title('PCA loading matrix (sign-corrected)')
plt.tight_layout()
plt.savefig('data/week4_day5_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Factor scores parquet

In [ ]:
scores = pd.read_parquet('../data/processed/factor_scores.parquet')
scores['date'] = pd.to_datetime(scores['date'])
scores = scores.set_index('date')

print(f'Shape: {scores.shape}   {scores.index[0].date()} -> {scores.index[-1].date()}')
print(f'Columns: {list(scores.columns)}')
print()
print('Daily score stats (bp):')
score_cols = [c for c in scores.columns if c.startswith('score_')]
scores[score_cols[:3]].describe().round(2)

## 5. Verify: loadings are orthonormal

`L @ L.T` should be the identity matrix. Diagonal = 1 (each row has unit norm);
off-diagonal = 0 (rows are orthogonal). This is the definition of PCA components.

In [ ]:
gram = L @ L.T
print('L @ L.T (should be identity):')
print(gram.round(6))

is_identity = np.allclose(gram, np.eye(len(L)), atol=1e-6)
print(f'\nOrthonormal: {"PASS" if is_identity else "FAIL"}')